In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
import os
from dotenv import load_dotenv

load_dotenv()

True

In [14]:

endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )

In [15]:
model=ChatHuggingFace(llm=endpoint)

In [16]:
class LLMState(TypedDict):
    question: str
    answer: str

In [17]:
def llm_qa(state: LLMState) -> LLMState:
    # extract the question from state
    question = state["question"]
    # form a prompt
    prompt = f"Answer the following question: {question}"

    # ask that question to LLM
    answer = model.invoke(prompt).content
    state["answer"] = answer
    return state

In [18]:
graph = StateGraph(LLMState)
graph.add_node("llm_qa", llm_qa)

In [20]:
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)
workflow = graph.compile()

In [22]:
initial_state = {"question": "What is the capital of France?"}
output = workflow.invoke(initial_state)
print(output)
print(output['answer'])



{'question': 'What is the capital of France?', 'answer': 'The capital of France is Paris.'}
The capital of France is Paris.
